In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

device = torch.device("cude" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.ToTensor()

train_data = MNIST(
    root = "data",
    train = True,
    download = True,
    transform = transform
)

test_data = MNIST(
    root = "data",
    train = False,
    download = True,
    transform = transform
)

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 64,
    shuffle = True
)

test_loader = DataLoader(
    dataset = test_data,
    batch_size = 64,
    shuffle = False
)

class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.feature = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(16*7*7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.feature(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_lose = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_lose += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_lose/len(train_loader):.4f}")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predection = outputs.argmax(dim=1)
        correct += (predection == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

torch.save(model.state_dict(), "simple_cnn_mnist.pth")
print("Model saved to simple_cnn_mnist.pth")

Using device: cpu


100.0%
100.0%
100.0%
100.0%


Epoch [1/3], Loss: 0.2970
Epoch [2/3], Loss: 0.0800
Epoch [3/3], Loss: 0.0576
Test Accuracy: 98.47%
Model saved to simple_cnn_mnist.pth
